In [132]:
import pandas as pd
import json

Used the BLASTx web interface with the ClusteredNR sequence database as the reference database.

In [133]:
ann_dict_simple = {'MG334.mbin.2_00182': 'hypothetical protein (1)',

 'ERR10897595.sbin.1_00593': 'hypothetical protein (3)',
 'ERR4421622.sbin.2_01193': 'XRE family transcriptional regulator',
 'ERR10897696.sbin.1_01201': 'DUF771 domain-containing protein',
 'ERR10897595.sbin.1_00598': 'hypothetical protein (6)',
 'ERR10897595.sbin.1_00591': 'MarR family transcriptional regulator',

'ERR10897979.sbin.2_00762': 'hypothetical protein (2)',
 'ERR10897595.sbin.1_01051': 'hypothetical protein (4)',
 'ERR10897583.sbin.2_00776': 'hypothetical protein (5)',

'SRR17635712.mbin.7_01059': 'Na+/H+ antiporter NhaC',

 'ERR10898139.sbin.1_00313': 'LacI family transcriptional regulator',

  'ERR10897723.sbin.1_00478': 'type II toxin-antitoxin system HicA family toxin',
 'ERR10897583.sbin.2_00437': 'type II toxin-antitoxin system HicB family antitoxin'}

In [134]:
import pandas as pd
import re

def parse_blast(file_path):
    """
    Parses a text file containing BLAST results.
    Includes every listed cluster member as an individual row.
    Filters: Percent Identity >= 50%, Query Coverage >= 80%, 
    Species != 'Lactobacillus crispatus'
    """
    
    all_rows = []
    
    try:
        with open(file_path, 'r') as f:
            content = f.read()
    except Exception as e:
        print(f"Error reading file: {e}")
        return pd.DataFrame()

    # Split the file into sections for each Query
    query_sections = re.split(r'Query #\d+:\s+', content)
    
    for section in query_sections:
        if not section.strip() or "Cluster:" not in section:
            continue
            
        lines = section.strip().split('\n')
        # Extract Gene Name from the first line of the section
        gene_name = lines[0].split()[0]
        
        # Identify individual Cluster blocks within this Query section
        # Clusters are separated by blank lines or specific headers
        cluster_blocks = re.split(r'\n(?=Cluster:)', section)
        
        for block in cluster_blocks:
            if "Cluster:" not in block:
                continue
                
            # 1. Extract Cluster-level statistics
            identity = 0.0
            coverage = 0.0
            e_value = "0.0"
            cluster_id_text = ""

            # Extract identity
            id_match = re.search(r'Percent Identity:\s+([\d\.]+)', block)
            if id_match: identity = float(id_match.group(1))
            
            # Extract coverage
            cov_match = re.search(r'Percent Coverage:\s+([\d\.]+)', block)
            if cov_match: coverage = float(cov_match.group(1))
            
            # Extract E-value
            eval_match = re.search(r'Evalue:\s+(\S+)', block)
            if eval_match: e_value = eval_match.group(1)
            
            # Extract full cluster name for reference
            name_match = re.search(r'Cluster:\s+(.*)', block)
            if name_match: cluster_id_text = name_match.group(1).strip()

            # 2. Extract Individual Cluster Members
            # Look for the member table starting after "cluster member(s):"
            member_section = re.search(r'\d+ cluster member\(s\):\nAccession.*?\n(.*?)(\n\n|\n>|\nCluster:|$)', block, re.DOTALL)
            
            if member_section:
                member_lines = member_section.group(1).strip().split('\n')
                for m_line in member_lines:
                    # Format is usually: Accession | Scientific Name | Common Name | Taxid
                    # Because scientific names can have spaces, we split and re-join
                    parts = m_line.split()
                    if len(parts) < 2: continue
                    
                    member_accession = parts[0]
                    # The species is usually the middle parts. 
                    # Note: This logic assumes the last part is Taxid and second-to-last is Common Name
                    member_species = " ".join(parts[1:-2]).strip()
                    taxid = parts[-1]


                    # Apply Filters
                    if identity >= 50.0 and coverage >= 80.0:
                        all_rows.append({
                            'gene': gene_name,
                            'hit_name': member_accession,
                            'cluster_name': cluster_id_text,
                            'hit_species': member_species,
                            'percent_identity': identity,
                            'query_coverage': coverage,
                            'e_value': e_value,
                            'taxid': taxid,

                        })

    columns = ['gene', 'hit_name', 'cluster_name', 'hit_species', 'percent_identity', 'query_coverage', 'e_value', 'taxid']
    return pd.DataFrame(all_rows, columns=columns)

file_name = 'BLAST_other_species/BLASTx_sig_gene_clusters.txt'
df = parse_blast(file_name)
df[['cluster_name', 'cluster_annotation']] = (
    df['cluster_name']
    .str.split(' [', n=1, expand=True, regex=False)[0]
    .str.split(' ', n=1, expand=True)
)

df = df.drop(columns='cluster_name')
df['cluster_annotation'] = df['cluster_annotation'].str.replace('MULTISPECIES: ','')
df['taxid'] = df['taxid'].astype(int)
df['gene_annotation'] = df['gene'].map(ann_dict_simple)
df.head()

,gene,hit_name,hit_species,percent_identity,query_coverage,e_value,taxid,cluster_annotation,gene_annotation
0,MG334.mbin.2_00182,WP_061205165.1,Lactobacillus crispatus,91.38,100.0,0.0,47770,hypothetical protein,hypothetical protein (1)
1,MG334.mbin.2_00182,MCH4005403.1,Lactobacillus crispatus,91.38,100.0,0.0,47770,hypothetical protein,hypothetical protein (1)
2,MG334.mbin.2_00182,MCI1523717.1,Lactobacillus crispatus,91.38,100.0,0.0,47770,hypothetical protein,hypothetical protein (1)
3,MG334.mbin.2_00182,MDU7065928.1,Lactobacillus crispatus,91.38,100.0,0.0,47770,hypothetical protein,hypothetical protein (1)
4,MG334.mbin.2_00182,MFJ6919607.1,Lactobacillus crispatus,91.38,100.0,0.0,47770,hypothetical protein,hypothetical protein (1)


In [135]:
!pwd

/Users/cdubin/Library/CloudStorage/Box-Box/VMGC_cervical_dysplasia_paper/code/cervical_dysplasia/microSLAM


In [136]:
df.shape, df['gene'].nunique()

((1661, 9), 13)

### Map taxids to taxonomic names

conda activate ncbi_datasets
cd BLAST_other_species
datasets summary taxonomy taxon --inputfile taxids.txt --as-json-lines | dataformat tsv taxonomy --template tax-summary > taxid_to_name.tsv


In [137]:
taxids = df['taxid'].unique()

with open('BLAST_other_species/taxids.txt', 'w') as f:
    for t in taxids:
        f.write(f'{t}\n')

In [138]:
tax_info = pd.read_csv('BLAST_other_species/taxid_to_name.tsv', sep='\t').set_index('Taxid')
tax_info.head()

,Query,Tax name,Authority,Rank,Basionym,Basionym authority,Curator common name,Has type material,Group name,Domain/Realm name,...,Class taxid,Order name,Order taxid,Family name,Family taxid,Genus name,Genus taxid,Species name,Species taxid,Scientific name is formal
Taxid,,,,,,,,,,,,,,,,,,,,,
1007676,1007676,Companilactobacillus ginsenosidimutans,Zheng et al. 2020,SPECIES,NaN,NaN,NaN,yes,firmicutes,Bacteria,...,91061.0,Lactobacillales,186826.0,Lactobacillaceae,33958.0,Companilactobacillus,2767879.0,Companilactobacillus ginsenosidimutans,1007676.0,True
1042399,1042399,Lactobacillus delbrueckii subsp. bulgaricus CN...,NaN,STRAIN,NaN,NaN,NaN,no,firmicutes,Bacteria,...,91061.0,Lactobacillales,186826.0,Lactobacillaceae,33958.0,Lactobacillus,1578.0,Lactobacillus delbrueckii,1584.0,False
104955,104955,Limosilactobacillus frumenti,(Müller et al. 2000) Zheng et al. 2020,SPECIES,Lactobacillus frumenti,Muller et al. 2000,NaN,yes,firmicutes,Bacteria,...,91061.0,Lactobacillales,186826.0,Lactobacillaceae,33958.0,Limosilactobacillus,2742598.0,Limosilactobacillus frumenti,104955.0,True
1054041,1054041,Levilactobacillus yonginensis,(Yi et al. 2013) Zheng et al. 2020,SPECIES,Lactobacillus yonginensis,Yi et al. 2013,NaN,yes,firmicutes,Bacteria,...,91061.0,Lactobacillales,186826.0,Lactobacillaceae,33958.0,Levilactobacillus,2767886.0,Levilactobacillus yonginensis,1054041.0,True
105612,105612,Dellaglioa algida,(Kato et al. 2000) Zheng et al. 2020,SPECIES,Lactobacillus algidus,Kato et al. 2000,NaN,yes,firmicutes,Bacteria,...,91061.0,Lactobacillales,186826.0,Lactobacillaceae,33958.0,Dellaglioa,2767880.0,Dellaglioa algida,105612.0,True


In [139]:
for gene, ann in ann_dict_simple.items():
    print(gene, ann)
    temp = df[df['gene']==gene]

    print(temp['cluster_annotation'].value_counts())
    print()


MG334.mbin.2_00182 hypothetical protein (1)
cluster_annotation
hypothetical protein    73
Name: count, dtype: int64

ERR10897595.sbin.1_00593 hypothetical protein (3)
cluster_annotation
hypothetical protein             20
hypothetical protein, partial     3
Name: count, dtype: int64

ERR4421622.sbin.2_01193 XRE family transcriptional regulator
cluster_annotation
helix-turn-helix domain-containing protein, partial    23
helix-turn-helix domain-containing protein             13
helix-turn-helix transcriptional regulator              8
Name: count, dtype: int64

ERR10897696.sbin.1_01201 DUF771 domain-containing protein
cluster_annotation
DUF771 domain-containing protein    113
Name: count, dtype: int64

ERR10897595.sbin.1_00598 hypothetical protein (6)
cluster_annotation
hypothetical protein                  29
hypothetical protein FC28_GL001748     1
Name: count, dtype: int64

ERR10897595.sbin.1_00591 MarR family transcriptional regulator
cluster_annotation
helix-turn-helix domain-contai

In [140]:
df = df.merge(tax_info, left_on='taxid', right_index=True, how='left')
df = df.drop(columns=['Scientific name is formal', 'Query', 'Authority','Domain/Realm taxid','cluster_annotation',
 'Kingdom taxid',
 'Phylum taxid',
 'Class taxid',
 'Order taxid',
 'Family taxid',
 'Genus taxid',
 'Species taxid'])
df[['Family name', 'Genus name','Species name']] = df[['Family name', 'Genus name','Species name']].fillna('').astype(str)


In [141]:
df.to_csv('/Users/cdubin/Library/CloudStorage/Box-Box/VMGC_cervical_dysplasia_paper/tables/table_S4.csv', index=False)

In [144]:
!pwd

/Users/cdubin/Library/CloudStorage/Box-Box/VMGC_cervical_dysplasia_paper/code/cervical_dysplasia/microSLAM


In [145]:
not_lc = df[(df['Species name']!='') & (~df['Species name'].str.contains('Lactobacillus crispatus'))]


for gene, desc in ann_dict_simple.items():

    print(gene)
    print(desc)

    temp = not_lc[not_lc['gene']==gene]
    print(temp.value_counts('Species name'))

    print()

[ann_dict_simple[i] for i in ann_dict_simple if i not in not_lc['gene'].values]


MG334.mbin.2_00182
hypothetical protein (1)
Species name
uncultured Lactobacillus sp.    1
Name: count, dtype: int64

ERR10897595.sbin.1_00593
hypothetical protein (3)
Species name
Lactobacillus helveticus    3
Name: count, dtype: int64

ERR4421622.sbin.2_01193
XRE family transcriptional regulator
Species name
Lactobacillus helveticus        3
Lactobacillus xujianguonis      1
Ligilactobacillus salivarius    1
Name: count, dtype: int64

ERR10897696.sbin.1_01201
DUF771 domain-containing protein
Species name
Limosilactobacillus reuteri       16
Lactobacillus gasseri              8
Lactobacillus paragasseri          7
Lactobacillus helveticus           5
Lactobacillus johnsonii            4
Lactobacillus melliventris         4
Limosilactobacillus portuensis     4
Lactobacillus xujianguonis         3
Lactobacillus sp.                  3
Lactobacillus taiwanensis          2
Caudoviricetes sp.                 2
uncultured Lactobacillus sp.       2
Lactobacillus kullabergensis       2
Limosil

['hypothetical protein (2)']

In [146]:
not_lactobacillaceae = df[df['Family name'] != 'Lactobacillaceae']


for gene, desc in ann_dict_simple.items():

    print(gene)
    print(desc)

    temp = not_lactobacillaceae[not_lactobacillaceae['gene']==gene]
    print(temp.value_counts('Species name'))

    print()


[ann_dict_simple[i] for i in ann_dict_simple if i not in not_lactobacillaceae['gene'].values]


MG334.mbin.2_00182
hypothetical protein (1)
Series([], Name: count, dtype: int64)

ERR10897595.sbin.1_00593
hypothetical protein (3)
Series([], Name: count, dtype: int64)

ERR4421622.sbin.2_01193
XRE family transcriptional regulator
Series([], Name: count, dtype: int64)

ERR10897696.sbin.1_01201
DUF771 domain-containing protein
Species name
Caudoviricetes sp.    2
                      1
Bacteriophage sp.     1
Name: count, dtype: int64

ERR10897595.sbin.1_00598
hypothetical protein (6)
Series([], Name: count, dtype: int64)

ERR10897595.sbin.1_00591
MarR family transcriptional regulator
Species name
Chlamydia trachomatis    1
Name: count, dtype: int64

ERR10897979.sbin.2_00762
hypothetical protein (2)
Series([], Name: count, dtype: int64)

ERR10897595.sbin.1_01051
hypothetical protein (4)
Species name
Bacteriophage sp.    1
Name: count, dtype: int64

ERR10897583.sbin.2_00776
hypothetical protein (5)
Species name
Bacteriophage sp.    1
Name: count, dtype: int64

SRR17635712.mbin.7_01059

['hypothetical protein (1)',
 'hypothetical protein (3)',
 'XRE family transcriptional regulator',
 'hypothetical protein (6)',
 'hypothetical protein (2)']